# Xây dựng mô hình dự đoán kết quả bệnh nhân mắc bệnh tiểu đường trên dataset Pima Indians Diabetes

## 1. Import các thư viện cần thiết

In [143]:
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
import seaborn as sns

# preprocessing
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# model selection
from sklearn.model_selection import StratifiedKFold, GridSearchCV, KFold

# algorithms
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
import sklearn

from xgboost import XGBClassifier

# metrics
from sklearn.metrics import accuracy_score , ConfusionMatrixDisplay, confusion_matrix, f1_score, roc_auc_score, classification_report

import warnings

## 2. Thiết lập tham số

In [144]:
parameters = {}
parameters["random_state"] = 42
parameters["k_fold"] = 10

## 3. Lấy dataset đã được dọn dẹp và chuẩn hóa.

In [145]:
X_train = pd.DataFrame(np.load("../exps/data/X_train_scaled.npy"))
y_train = pd.DataFrame(np.load("../exps/data/y_train.npy"))
X_train = X_train.squeeze()
y_train = y_train.squeeze()

X_test = pd.DataFrame(np.load("../exps/data/X_test_scaled.npy"))
y_test = pd.DataFrame(np.load("../exps/data/y_test.npy"))
X_test = X_test.squeeze()
y_test = y_test.squeeze()

## 4. Kiểm tra dữ liệu

In [146]:
X_train.head()

,0,1,2,3,4,5
0,0.091816,1.509134,0.232332,-0.855792,0.060909,-0.937699
1,0.819760,0.166432,0.275548,0.219850,0.060909,0.923014
2,-0.282398,1.580089,1.634622,-0.855792,1.192643,0.662844
3,1.721401,1.028798,-0.810374,0.951417,1.319553,0.917181
4,-1.711465,2.107460,1.272717,-0.381319,-0.428882,0.200399


In [147]:
y_train.head()

0    1
1    0
2    1
3    1
4    1
Name: 0, dtype: int64

In [148]:
X_test.head()

,0,1,2,3,4,5
0,-1.711465,1.917549,0.374947,0.802700,2.136763,-0.030104
1,0.382080,0.166432,0.040584,0.309908,-0.780422,-0.825212
2,-0.809824,0.422981,-1.397126,-1.264453,-0.601948,-0.194678
3,0.619242,0.199392,0.332591,0.565790,0.215262,-0.849994
4,0.091816,0.974805,-1.971610,0.482751,0.365553,-0.904107


In [149]:
y_test.head()

0    1
1    1
2    0
3    1
4    0
Name: 0, dtype: int64

## 5. Chia K-Fold

In [ ]:
skf = StratifiedKFold(n_splits=parameters["k_fold"], shuffle=True, random_state=parameters["random_state"])

## 6. Chọn các model để train.
Các model được chọn bao gồm: Logistic Regression, Random Forest, SVC, K-Neighbours, Decision Tree.

In [151]:
models = {}
models["Logistic Regression"] = LogisticRegression(class_weight="balanced", 
                                                  max_iter=1000,
                                                  random_state=parameters["random_state"])
                                
models["Random Forest"] = RandomForestClassifier(
        class_weight="balanced", 
        random_state=parameters["random_state"], 
        n_jobs=-1, 
        max_depth=7,
        min_samples_leaf=3,
        n_estimators=200)

models["SVC"] = SVC(
        kernel='rbf',         
        class_weight='balanced', 
        probability=True,
        random_state=parameters["random_state"]
    )

models["K-Neighbours"] = KNeighborsClassifier(n_neighbors=5, weights='distance')

models["Decision Tree"] = DecisionTreeClassifier(max_depth=7,
                                                 random_state=parameters["random_state"],
                                                 min_samples_leaf=3)


## 7. Train các model trên tập dữ liệu train.

In [152]:
results = {name: {'accuracy': [], 'f1': [], 'roc_auc': []} for name in models.keys()}

for train_index, val_index in skf.split(X_train, y_train):
    X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

    for name, model in models.items():
        model.fit(X_train_fold, y_train_fold)

        y_pred = model.predict(X_val_fold)    
        y_prob = model.predict_proba(X_val_fold)[:,1]

        acc = accuracy_score(y_val_fold, y_pred)
        f1 = f1_score(y_val_fold, y_pred)
        roc_auc = roc_auc_score(y_val_fold, y_prob)

        results[name]['accuracy'].append(acc)
        results[name]['f1'].append(f1)
        results[name]['roc_auc'].append(roc_auc)


print("--------------------------")
for name in models.keys():
    acc_mean = sum(results[name]['accuracy']) / len(results[name]['accuracy'])
    f1_mean = sum(results[name]['f1']) / len(results[name]['f1'])
    roc_auc_mean = sum(results[name]['roc_auc']) / len(results[name]['roc_auc'])
    print(f"{name} Average Results:")
    print(f"  Accuracy: {acc_mean:.4f}")
    print(f"  F1 Score: {f1_mean:.4f}")
    print(f"  ROC AUC:  {roc_auc_mean:.4f}")
    print("-" * 30)
    

--------------------------
Logistic Regression Average Results:
  Accuracy: 0.7499
  F1 Score: 0.6687
  ROC AUC:  0.8356
------------------------------
Random Forest Average Results:
  Accuracy: 0.7653
  F1 Score: 0.6886
  ROC AUC:  0.8346
------------------------------
SVC Average Results:
  Accuracy: 0.7622
  F1 Score: 0.6983
  ROC AUC:  0.8387
------------------------------
K-Neighbours Average Results:
  Accuracy: 0.7469
  F1 Score: 0.6132
  ROC AUC:  0.8111
------------------------------
Decision Tree Average Results:
  Accuracy: 0.7055
  F1 Score: 0.5462
  ROC AUC:  0.7261
------------------------------


## 8. Kiểm tra hiệu suất model trên tập test.

In [153]:
for name, model in models.items():
    
    y_pred = model.predict(X_test)    
    y_prob = model.predict_proba(X_test)[:,1]

    print(f"-------------{name}------------")

    print("\nPredicted values (first 10):")
    print(y_pred[:10].tolist())

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


-------------Logistic Regression------------

Predicted values (first 10):
[1, 0, 0, 1, 0, 0, 1, 0, 1, 0]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.74      0.80        76
           1       0.62      0.80      0.70        40

    accuracy                           0.76       116
   macro avg       0.75      0.77      0.75       116
weighted avg       0.79      0.76      0.76       116

Confusion Matrix:
[[56 20]
 [ 8 32]]
-------------Random Forest------------

Predicted values (first 10):
[1, 0, 0, 0, 0, 0, 1, 0, 1, 0]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.80      0.82        76
           1       0.66      0.72      0.69        40

    accuracy                           0.78       116
   macro avg       0.75      0.76      0.76       116
weighted avg       0.78      0.78      0.78       116

Confusion Matrix:
[[61 15]
 [11 29]]
-------------SVC--

## 9. Lưu file dưới dạng HTML.

In [154]:
import nbformat
from nbconvert import HTMLExporter

# load notebook
nb = nbformat.read("model.ipynb", as_version=4)

# tạo exporter
html_exporter = HTMLExporter()
html_exporter.template_name = 'lab'

(body, resources) = html_exporter.from_notebook_node(nb)

# ghi file HTML
with open("../exps/model1/model2.html", "w", encoding="utf-8") as f:
    f.write(body)

# Kết thúc